# 🕳️ Detecção de Buracos em Vias Urbanas — Treino YOLOv8
**Global Solution 2026.2 · FIAP · Cognitive Computing · CitizenTech**

---
## ⚠️ Antes de começar
1. Vá em **Runtime → Change runtime type → GPU → T4** e salve.
2. Execute as células **em ordem**, uma por vez.
3. Ao final, **baixe `best.pt`** para a pasta `models/` do repositório local.

**Dataset:** Pothole Detection — Intel Unnati Training Program (Roboflow Universe)  
**Licença:** MIT  
**Modelo base:** YOLOv8n (nano) — rápido para treinar na T4 e leve para inferência na CPU do lab

## Célula 1 — Verificar GPU

In [ ]:
# Verifica se a GPU T4 está disponível.
# Se aparecer 'NVIDIA T4', está tudo certo.
# Se aparecer 'No GPU', vá em Runtime > Change runtime type > GPU.
import subprocess
resultado = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(resultado.stdout if resultado.returncode == 0 else '❌ GPU não encontrada — altere o runtime!')

## Célula 2 — Instalar dependências

In [ ]:
# Instala o Ultralytics (framework YOLOv8) e o cliente Roboflow.
# O Colab já tem torch/cuda instalados; só precisamos do ultralytics e roboflow.
# A flag -q suprime a saída verbosa do pip para não poluir o notebook.
%pip install ultralytics roboflow -q
print('✅ Dependências instaladas.')

## Célula 3 — Baixar o dataset do Roboflow

In [ ]:
# Baixa o dataset 'Pothole Detection' (Intel Unnati Training Program)
# diretamente do Roboflow Universe em formato YOLOv8.
#
# COMO OBTER A API KEY:
#   1. Acesse https://universe.roboflow.com e crie uma conta gratuita.
#   2. Vá em Settings > API Keys e copie sua chave.
#   3. Cole abaixo no lugar de 'SUA_API_KEY_AQUI'.
#
# O dataset será salvo em /content/datasets/pothole-detection-*/
# O data.yaml é gerado automaticamente pelo Roboflow.

from roboflow import Roboflow

API_KEY = 'SUA_API_KEY_AQUI'  # ← substitua pela sua chave

rf = Roboflow(api_key=API_KEY)

# Workspace e projeto do Intel Unnati no Roboflow Universe
projeto = rf.workspace('intel-unnati-training-program').project('pothole-detection-bqu6s')

# Baixa a versão mais recente em formato YOLOv8
dataset = projeto.version(1).download('yolov8')

print(f'✅ Dataset baixado em: {dataset.location}')
print(f'   data.yaml: {dataset.location}/data.yaml')

## Célula 4 — Inspecionar o dataset

In [ ]:
# Conta imagens e rótulos por split para confirmar que o download foi correto.
# Esperado: ~2.000 imagens de treino, ~500 de validação, ~300 de teste.
# Qualquer split com 0 imagens indica problema no download.

import os
from pathlib import Path

DATA_DIR = Path(dataset.location)
YAML_PATH = DATA_DIR / 'data.yaml'

print(f'Raiz do dataset: {DATA_DIR}\n')

extensoes = {'.jpg', '.jpeg', '.png', '.bmp'}
for split in ['train', 'valid', 'test']:
    img_dir = DATA_DIR / split / 'images'
    lbl_dir = DATA_DIR / split / 'labels'
    if img_dir.exists():
        imgs = [p for p in img_dir.rglob('*') if p.suffix.lower() in extensoes]
        lbls = list(lbl_dir.rglob('*.txt')) if lbl_dir.exists() else []
        print(f'  {split:6s}: {len(imgs):4d} imagens | {len(lbls):4d} rótulos')
    else:
        print(f'  {split:6s}: ❌ não encontrado')

# Exibe o conteúdo do data.yaml gerado pelo Roboflow
print('\n--- data.yaml ---')
print(YAML_PATH.read_text())

## Célula 5 — Visualizar amostras do dataset

In [ ]:
# Exibe 6 imagens aleatórias do conjunto de treino com as bounding boxes
# desenhadas a partir dos rótulos YOLO.
#
# Por que visualizar antes de treinar:
#   Detecta problemas de anotação (caixas desalinhadas, classes erradas)
#   que são invisíveis nos números mas destroem a qualidade do treino.

import random
import cv2
import matplotlib.pyplot as plt
import numpy as np

def desenhar_bbox_yolo(img_path: Path, lbl_path: Path) -> np.ndarray:
    """Desenha as bounding boxes YOLO normalizadas sobre a imagem."""
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if lbl_path.exists():
        for linha in lbl_path.read_text().strip().splitlines():
            partes = linha.split()
            if len(partes) == 5:
                _, cx, cy, bw, bh = map(float, partes)
                x1 = int((cx - bw/2) * w)
                y1 = int((cy - bh/2) * h)
                x2 = int((cx + bw/2) * w)
                y2 = int((cy + bh/2) * h)
                cv2.rectangle(img, (x1, y1), (x2, y2), (255, 50, 50), 2)
    return img

img_dir = DATA_DIR / 'train' / 'images'
lbl_dir = DATA_DIR / 'train' / 'labels'
amostras = random.sample(list(img_dir.glob('*.*')), min(6, len(list(img_dir.glob('*.*')))))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Amostras do dataset de treino (bounding boxes em vermelho)', fontsize=13)
for ax, img_path in zip(axes.flat, amostras):
    lbl_path = lbl_dir / (img_path.stem + '.txt')
    img = desenhar_bbox_yolo(img_path, lbl_path)
    ax.imshow(img)
    ax.set_title(img_path.name, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.savefig('amostras_dataset.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Grade salva como amostras_dataset.png')

## Célula 6 — Configurar e iniciar o treino

In [ ]:
# Treina YOLOv8n (nano) a partir dos pesos pré-treinados no COCO.
#
# Hiperparâmetros escolhidos:
#   model  : yolov8n.pt — menor variante do YOLOv8; ideal para GPU T4 e
#             para inferência na CPU do lab (boa velocidade sem GPU).
#   epochs : 50 — suficiente para convergir num dataset de ~2k imagens;
#             aumentar para 100 se o val/mAP ainda estiver subindo.
#   imgsz  : 640 — resolução padrão do YOLO; boa relação custo/benefício.
#   batch  : 16 — cabe na VRAM da T4 (16 GB) com folga.
#   patience: 15 — para o treino se mAP50 não melhorar em 15 épocas
#              (early stopping), economizando tempo.
#   project/name: organiza os artefatos do treino em runs/detect/gs2026/

from ultralytics import YOLO

modelo = YOLO('yolov8n.pt')  # carrega pesos pré-treinados no COCO (transfer learning)

resultados = modelo.train(
    data    = str(YAML_PATH),
    epochs  = 50,
    imgsz   = 640,
    batch   = 16,
    patience= 15,
    device  = 0,           # GPU 0 (T4)
    project = 'runs/detect',
    name    = 'gs2026',
    exist_ok= True,        # sobrescreve se rodar novamente
    verbose = True,
)

print('\n✅ Treino concluído!')
print(f'   Melhor modelo salvo em: runs/detect/gs2026/weights/best.pt')

## Célula 7 — Avaliar no conjunto de teste (val rápido)

In [ ]:
# Avaliação rápida no conjunto de validação usando o melhor modelo salvo.
# A avaliação completa (com todos os gráficos) é feita pelo src/evaluate.py
# LOCALMENTE após baixar o best.pt.
#
# Aqui apenas confirmamos que o modelo treinado funciona e vemos as
# métricas principais antes de baixar.

from ultralytics import YOLO

melhor = YOLO('runs/detect/gs2026/weights/best.pt')

metricas = melhor.val(
    data   = str(YAML_PATH),
    split  = 'val',
    imgsz  = 640,
    device = 0,
)

print(f'\n📊 Resultados no conjunto de validação:')
print(f'   Precisão  (P):  {metricas.box.mp:.4f}')
print(f'   Recall    (R):  {metricas.box.mr:.4f}')
print(f'   mAP@50:         {metricas.box.map50:.4f}')
print(f'   mAP@50-95:      {metricas.box.map:.4f}')

## Célula 8 — Visualizar predições de exemplo

In [ ]:
# Roda inferência em 6 imagens do conjunto de validação e exibe
# as bounding boxes previstas pelo modelo treinado.
#
# conf=0.25: limiar de confiança (descartar predições abaixo de 25%).
# iou=0.45:  limiar de IoU para NMS (Non-Maximum Suppression).

import random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from ultralytics import YOLO

melhor = YOLO('runs/detect/gs2026/weights/best.pt')

val_imgs = list((DATA_DIR / 'valid' / 'images').glob('*.*'))
amostras = random.sample(val_imgs, min(6, len(val_imgs)))

resultados_pred = melhor.predict(
    source = amostras,
    conf   = 0.25,
    iou    = 0.45,
    save   = True,
    project= 'runs/predict',
    name   = 'exemplos',
    exist_ok=True,
)

# Exibe os resultados salvos
pred_dir = Path('runs/predict/exemplos')
imgs_salvas = sorted(pred_dir.glob('*.jpg'))[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Predições do modelo treinado (conf ≥ 0.25)', fontsize=13)
for ax, p in zip(axes.flat, imgs_salvas):
    ax.imshow(mpimg.imread(str(p)))
    ax.set_title(p.name, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.savefig('predicoes_exemplo.png', dpi=120, bbox_inches='tight')
plt.show()
print('✅ Grade de predições salva como predicoes_exemplo.png')

## Célula 9 — Baixar os artefatos

In [ ]:
# Compacta e baixa os artefatos necessários para continuar localmente:
#   best.pt          → salvar em models/
#   amostras_dataset.png e predicoes_exemplo.png → salvar em examples/
#
# Após baixar, coloque best.pt em models/ no repositório local
# e rode src/evaluate.py para gerar as métricas completas.

import shutil
from google.colab import files
from pathlib import Path

# Cria pasta temporária para os artefatos
artefatos_dir = Path('artefatos_gs2026')
artefatos_dir.mkdir(exist_ok=True)

# Copia o melhor modelo
shutil.copy('runs/detect/gs2026/weights/best.pt', artefatos_dir / 'best.pt')

# Copia as imagens de exemplo
for img in ['amostras_dataset.png', 'predicoes_exemplo.png']:
    if Path(img).exists():
        shutil.copy(img, artefatos_dir / img)

# Compacta tudo
shutil.make_archive('artefatos_gs2026', 'zip', artefatos_dir)

# Inicia o download
files.download('artefatos_gs2026.zip')

print('✅ Download iniciado!')
print('   Descompacte e mova:')
print('   - best.pt            → models/')
print('   - *.png              → examples/')

## ✅ Próximos passos (localmente, no seu repositório)

```bash
# 1. Validar dataset e gerar data.yaml local
python src/dataset.py --data_dir data/ --output data/data.yaml

# 2. Avaliar e gerar métricas completas
python src/evaluate.py --weights models/best.pt --data data/data.yaml

# 3. Analisar erros
python src/analise_erros.py --weights models/best.pt --data data/data.yaml

# 4. Demo ao vivo (imagem)
python src/demo.py --source examples/foto.jpg --weights models/best.pt
```

---
*Notebook gerado para o projeto de Cognitive Computing — FIAP GS 2026.2*